# An adaptive integration program

In [ ]:
#    APM41012EP course notebook - Chapter 3 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Adaptive integration based on Gauss-Legendre 
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np

import plotly.graph_objs as go

from scipy.special import legendre, roots_legendre
from sympy.integrals.quadrature import gauss_legendre
import warnings
warnings.filterwarnings('ignore')

## Construction of an adaptive quadrature

The aim of this additional section is to see how one can construct an adaptive quadrature based on Gauss quadrature.
More precisely,  for a given function $f:[a, b] \rightarrow \mathbb{R}$, we seek to compute 
the value of $\int_{a}^{b} f(x) {\mathrm d}x$ to a relative accuracy of $\mathrm{rTol}$. If the quadrature formula is fixed (for example the order-29 Gauss formula with $s=15$ ), 
one has to find a division $\Delta=\left\{a=x_{0}<x_{1}<\ldots<x_{N}=b\right\}$ of the interval $(a, b)$ such that the numerical approximation via the composite formula based on  the 15-point order-29 Gauss quadrature formula, 
$I_{\Delta}^{15}$ satisfies:

$$
\left|I_{\Delta}^{15}-\int_{a}^{b} f(x) {\mathrm d}x\right| \leq \mathrm{rTol}  \int_{a}^{b}|f(x)| {\mathrm d}x
$$

For a function $f$ that does not change sign on $(a, b)$, the previous condition means that the relative error in the sense of the infinity norm is bounded by $\mathrm{rTol}$.
To construct this adaptive quadrature, we are  faced with two problems:
 * the choice of the division so that  the previous inequality is satisfied;
 * the construction of an estimate of the error $I_{\Delta}-\int_{a}^{b} f(x) {\mathrm d}x$.

### Determination of the division

For a sub-interval $\left(x_{0}, x_{0}+h\right)$ of $(a, b),$ we know how to compute the values:

$$
\operatorname{res}\left(x_{0}, x_{0}+h\right) =h \sum_{i=0}^{s-1} b_{i} f\left(x_{0}+c_{i} h\right) \\
\text {resabs}\left(x_{0}, x_{0}+h\right) =h \sum_{i=0}^{s-1} b_{i}\left|f\left(x_{0}+c_{i} h\right)\right|
$$

Let us assume, for the moment, that we also know an estimate of the error

$$
\operatorname{err}\left(x_{0}, x_{0}+h\right) \approx \operatorname{res}\left(x_{0}, x_{0}+h\right)-\int_{x_{0}}^{x_{0}+h} f(x) {\mathrm d}x
$$

The algorithm to find a suitable division is the following:

(i) we compute $\operatorname{res}(a, b), \operatorname{resabs}(a, b)$ and $\operatorname{err}(a, b) .$ If

$$
|\operatorname{err}(a, b)| \leq \mathrm{rTol}  \operatorname{resabs}(a, b)
$$

(ii) we accept $\operatorname{res}(a, b)$ as an approximation of $\int_{a}^{b} f(x) {\mathrm d}x$ and we stop the computation; otherwise

(iii) we subdivide $(a, b)$ into two sub-intervals $I_{1}=(a,(a+b) / 2)$ and $I_{2}=((a+b) / 2, b)$ and we compute $\operatorname{res}\left(I_{1}\right),$ resabs $\left(I_{1}\right), \operatorname{err}\left(I_{1}\right)$ and $\operatorname{res}\left(I_{2}\right),$ resabs $\left(I_{2}\right), \operatorname{err}\left(I_{2}\right) .$ We set $N=2$ and we
check whether:

$$
\sum_{j=1}^{N}\left|\operatorname{err}\left(I_{j}\right)\right| \leq \operatorname{\mathrm{rTol}} \left(\sum_{j=1}^{N} \operatorname{resabs}\left(I_{j}\right)\right).
$$

If the previous inequality holds, we accept $\operatorname{res}\left(I_{1}\right)+\operatorname{res}\left(I_{2}\right)$ as an approximation of $\int_{a}^{b} f(x) {\mathrm d}x ;$ otherwise

we set $N:=N+1$ and we subdivide the interval where the error is maximal (say $I_{k}$ ) into two equidistant sub-intervals, which we denote by $I_{k}$ and $I_{N+1}$. Then we compute res, resabs and err for these two intervals. If the previous inequality holds, we stop the computation and we accept

$$
\sum_{j=1}^{N} \operatorname{res}\left(I_{j}\right) \approx \int_{a}^{b} f(x) {\mathrm d}x
$$

as an approximation of the integral; otherwise we repeat part (iii) of this algorithm.

### Error estimation
 
 Unfortunately, the error formulas obtained at the beginning of the course are not very useful  for this new objective, because  the $p^{\text {th }}$ derivative of the function $f(x)$ is only rarely known, in particular 
 when $p=29$!
The idea is to apply another quadrature formula of lower order $\left(\widehat{b}_{i}, \widehat{c}_{i}\right) \hat{s}_{i=1}$ and to use the difference of the two numerical approximations as an estimate of the error of the  less accurate
result. 
So that the additional work is very small, we assume $\widehat{s} \leq s$ and we reuse the same evaluations of $f$, that is, we assume $\widehat{c}_{i}=c_{i}$ for all $i$. 

Such a quadrature formula is called an embedded formula if, for at least one index $i$, we have $\hat{b}_{i} \neq b_{i}$.
If $\left(b_{i}, c_{i}\right)_{i=0}^{s-1}$ is a quadrature formula of order $p \geq s,$ the order of an embedded formula is $\widehat{p} \leq s-1 .$ This result follows from the fact that, for a quadrature formula of order $\geq s,$ the weights $b_{i}$ are uniquely determined by its nodes $c_{i}$.

For the Gauss formula $(s=15, p=29),$ we obtain an embedded formula $\left(\widehat{b}_{i}, c_{i}\right)_{i=0}^{s-1}$ of order 13 by removing the midpoint $c_{8}=1 / 2$, i.e., by setting $\hat{b}_{8}=0$. 
The computable expression:

$$
\mathrm{err}_1:=h \sum_{i=0}^{s-1} b_{i} f\left(x_{0}+c_{i} h\right)-h \sum_{i=0}^{s-1} \widehat{b}_{i} f\left(x_{0}+c_{i} h\right) \approx C_{1} h^{15}
$$

is an approximation of the error of the embedded formula, because

$$
\left(h \sum_{i=1}^{s} b_{i} f\left(x_{0}+c_{i} h\right)-\int_{x_{0}}^{x_{0}+h} f(x) d x\right)+\left(\int_{x_{0}}^{x_{0}+h} f(x) d x-h \sum_{i=1}^{s} \widehat{b}_{i} f\left(x_{0}+c_{i} h\right)\right)
$$

with an estimate of the first term in $C h^{31}+\mathcal{O}\left(h^{32}\right)$ and of the second in $C h^{15}+\mathcal{O}\left(h^{16}\right)$.

In order to carry out the adaptation, we consider a second embedded formula whose nodes are $\left\{c_{2}, c_{4}, c_{6}, c_{10}, c_{12}, c_{14}\right\}$ and whose order is 5.
The weights of this quadrature formula are denoted by $\hat{\hat{b}_{i}}$ and we define:

$$
\mathrm{err}_2:=h \sum_{i=0}^{s-1} b_{i} f\left(x_{0}+c_{i} h\right)-h \sum_{i=0}^{s-1} \widehat{\widehat{b}}_{i} f\left(x_{0}+c_{i} h\right) \approx C_{2} h^{7}.
$$

There are several possibilities for defining $\operatorname{err}\left(x_{0}, x_{0}+h\right)$:
* $\operatorname{err}\left(x_{0}, x_{0}+h\right):= \mathrm{err}_1$; this estimate is too pessimistic. In general, the Gauss formula gives a much better result than the order-$13$ embedded formula.
* to obtain a relatively optimal estimate, we have chosen the approximation

$$
\operatorname{err}\left(x_{0}, x_{0}+h\right):=\mathrm{err}_1 \left(\frac{\mathrm{err}_1}{\mathrm{err}_2}\right)^{2}, \quad\left(\approx h^{15} \left(\frac{h^{15}}{h^{7}}\right)^{2} \approx h^{31}\right),
$$

which gives very good results.

**Abscissas of the 15-point Gauss formula and of its embedded formulas with 14 and 6 points:**

In [ ]:
p = legendre(15)

p29 = (np.sort(p.r)+1.)/2
p13 = np.delete(p29, 7)
p05 = p13[[1, 3, 5, 8, 10, 12]]

fig = go.Figure()
fig.add_trace(go.Scatter(x=p29, y=np.zeros(p29.size), mode='markers', name='Gauss formula (order 29)'))
fig.add_trace(go.Scatter(x=p13, y=np.zeros(p13.size)+0.2, mode='markers', name='Embedded formula (order 13)'))
fig.add_trace(go.Scatter(x=p05, y=np.zeros(p05.size)+0.4, mode='markers', name='Embedded formula (order 5)'))
fig.update_yaxes(range=[-0.1, 0.5])
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.85))
fig.show()

**Weights of the 15-point Gauss formula (sympy - arbitrary precision):**

In [ ]:
# 15 Gauss points, with a precision of 20 significant digits
x, w = gauss_legendre(15, 20)
print("Roots of the Legendre polynomial on [-1,1]:")
print(x)
print("Weights:")
print(w)

x = np.array(x, dtype=float)
w = np.array(w, dtype=float)
print(x.size)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=np.arange(15)))
#fig.add_trace(go.Scatter(x=x, y=x, mode='lines+markers', line_dash='dash', name="Gauss-Legendre - 15pts", showlegend=True))
#fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.25))
#fig.show()

In [ ]:
def integral_step(f, xmin, xmax, r, w):
    h = xmax-xmin
    return h*np.sum(w * f(xmin+h*r))

def gauss(f, xmin, xmax, s, n):
    r, w = roots_legendre(s)
    r = 0.5*(r+1)
    w = 0.5*w
    x = np.linspace(xmin, xmax, n+1)
    res = 0.
    for i in range(n):
        res += integral_step(f, x[i], x[i+1], r, w)
    return res

def adapt(f, xmin, xmax, res_ref, tol=1.e-10, n_max=50, verbose=False):

    # computation of the abscissas and weights of the Gauss-Legendre method of order 30 with 15 stages
    r_30, w_30 = roots_legendre(15)
    r_30 = 0.5*(r_30+1.)
    w_30 = 0.5*w_30
    
    # computation of the abscissas and weights of the embedded method of order 6 with 6 stages
    r_06 = r_30[[1,3,5,9,11,13]]
    a = np.array([r_06**0, r_06**1, r_06**2, r_06**3, r_06**4, r_06**5])
    b = 1./np.arange(1,7)
    w_06 = np.linalg.solve(a, b)
    
    # computation of the abscissas and weights of the embedded method of order 14 with 14 stages
    r_14 = r_30[[0,1,2,3,4,5,6,8,9,10,11,12,13,14]]
    a = np.array([r_14**0, r_14**1, r_14**2, r_14**3,  r_14**4,  r_14**5,  r_14**6, 
                  r_14**7, r_14**8, r_14**9, r_14**10, r_14**11, r_14**12, r_14**13])
    b = 1./np.arange(1,15)
    w_14 = np.linalg.solve(a, b)
    
    def adapt_step(xmin, xmax):
        
        h = xmax-xmin
        f_30 = f(xmin+h*r_30)
        
        res_30 =  h*np.sum(w_30 * f_30)
        res_14 =  h*np.sum(w_14 * f_30[[0,1,2,3,4,5,6,8,9,10,11,12,13,14]])
        res_06 =  h*np.sum(w_06 * f_30[[1,3,5,9,11,13]])
 
        err_1 = res_30 - res_14 
        err_2 = res_30 - res_06 
        err_est = err_1 * (err_1/err_2)**2
        
        return res_30, err_est 

    # initialization 
    res, err_est = adapt_step(xmin, xmax)
    #print(res,res_ref) 
    err_previous = abs(res - res_ref)
    #print(err_previous)
    
    if (np.abs(err_est) < tol*np.abs(res)):
        return res

    n = 2
 
    err_est_array = np.zeros(n_max)
    res_array = np.zeros(n_max)
  
    a = xmin
    b = xmax

    div = []
    div.append((a, (a+b)/2))
    div.append(((a+b)/2, b))
    
    div_per_ite = []
    div_per_ite.append(div.copy())
        
    ierrmax = 0
    
    while (n <= n_max):
                        
        res_array[ierrmax], err_est_array[ierrmax] = adapt_step(a, (a+b)/2)
        res_array[n-1], err_est_array[n-1] = adapt_step((a+b)/2, b)
                
        absres = np.sum(np.abs(res_array))
        res = np.sum(res_array)
        err_est = np.sum(np.abs(err_est_array))
        err     = np.abs(res - res_ref)
 
        if (verbose):
            print(f"It={n-1:2d}: res={res:14.12f}, |err|={err:16.10e}, |err est.|={err_est:16.10e}") 
            print(f"                            |err est.|/absres={err_est/absres:16.10e}, errN/errN-1 ={err/err_previous:6.5f}") 
        
        if (err_est < tol*absres):
            break
        
        err_previous = err
        
        ierrmax = np.argmax(np.abs(err_est_array)) 
        a, b = div[ierrmax]
        div[ierrmax] = (a, (a+b)/2)
        div.append(((a+b)/2, b))
            
        div_per_ite.append(div.copy()) 
        
        n += 1
    
    return res, div_per_ite

## First example

We consider the function:

$$ f(x) = 2 + \sin(3 \, cos(0.002(x-40)^2)) \quad \text{on }[10,110] $$

In [ ]:
def f(x):
    return 2 + np.sin(3*np.cos(0.002*(x-40)**2))

x = np.linspace(10, 110, 2000)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=f(x)))

fig.show()

**Application of the adaptation algorithm:**

In [ ]:
res_ref = gauss(f, 10, 110, 30, 10000)

print("Adaptive algorithm:")
res, div_per_ite = adapt(f, 10., 110., res_ref, tol=1.e-10, n_max=50, verbose=True)

# Note that N = number of iterations + 1 = number of intervals
print(f"\nNumber of iterations (subdivision by 2): {len(div_per_ite)} to satisfy |estimated err|/absres < tol, and {len(div_per_ite)+1} intervals")

In [ ]:
x = np.linspace(10, 110, 2000)

fig = go.Figure()

for ite in range(1, len(div_per_ite)+1):
    for div in div_per_ite[ite-1]:
        xdiv = np.linspace(div[0], div[1], 100)
        fig.add_trace(go.Scatter(visible=False, x=xdiv, y=f(xdiv), fill='tozeroy', showlegend=False))

        
# Make plot visible for ite=1
for i in range(len(div_per_ite[0])): fig.data[i].visible = True
    
# Create and add slider
ivis = 0
steps = []
for ite in range(1, len(div_per_ite)+1):
    step = dict(method="update", label = f" {ite}", args=[{"visible": [False] * len(fig.data)}])
    for idiv, div in enumerate(div_per_ite[ite-1]):
        #print(idiv, ivis)
        step["args"][0]["visible"][ivis] = True
        ivis = ivis+1
    steps.append(step)

sliders = [dict(currentvalue={"prefix": "ite: "}, steps=steps)]

fig.update_layout(sliders=sliders)
fig.update_xaxes(tickmode = 'array', tickvals = [div[0] for div in div_per_ite[-1]] + [110])

fig.show()

## Second example

We consider the function:

$$ g(x) = \sqrt{x} \, \log{x} \quad \text{on }]0,1] $$

In [ ]:
def g(x):
    return  np.sqrt(x)*np.log(x)

x = np.linspace(1.e-10, 1, 10000)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=g(x)))

fig.show()

**Application of the adaptation algorithm:**

In [ ]:
res_ref =  -4/9

print("Adaptive algorithm:")
res, div_per_ite = adapt(g, 1.e-20, 1, res_ref, tol=1.e-13, n_max=50, verbose=True)
print(f"\nNumber of iterations to satisfy |estimated err|/absres < tol: {len(div_per_ite)}")

In [ ]:
x = np.linspace(1.e-10, 1, 10000)

fig = go.Figure()

for ite in range(1, len(div_per_ite)+1):
    for div in div_per_ite[ite-1]:
        xdiv = np.linspace(div[0], div[1], 100)
        fig.add_trace(go.Scatter(visible=False, x=xdiv, y=g(xdiv), fill='tozeroy', showlegend=False))

# Make plot visible for ite=1
for i in range(len(div_per_ite[0])): fig.data[i].visible = True
    
# Create and add slider
ivis = 0
steps = []
for ite in range(1, len(div_per_ite)+1):
    step = dict(method="update", label = f" {ite}", args=[{"visible": [False] * len(fig.data)}])
    for idiv, div in enumerate(div_per_ite[ite-1]):
        #print(idiv, ivis)
        step["args"][0]["visible"][ivis] = True
        ivis = ivis+1
    steps.append(step)

sliders = [dict(currentvalue={"prefix": "ite: "}, steps=steps)]

fig.update_layout(sliders=sliders)

fig.show()